In [2]:
# Project 4: Diabetes Population Health Management
# Notebook 3: Prevalence Analysis and Visualization
# Objective: Create publication-quality visualizations and tables

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Environment loaded successfully")

# Load processed data
df = pd.read_csv('../data/processed/brfss_2023_diabetes.csv')

# Load weighted prevalence results from Notebook 2
with open('../outputs/tables/weighted_prevalence.json', 'r') as f:
    prev_results = json.load(f)

print(f"Loaded: {len(df):,} respondents")
print("Weighted prevalence results loaded")

# Create outputs directory structure
os.makedirs('../outputs/figures', exist_ok=True)
os.makedirs('../outputs/tables', exist_ok=True)

print("Output directories ready")

# Extract prevalence data for visualization
overall_prev = prev_results['overall']['prevalence']
overall_ci_lower = prev_results['overall']['ci_lower']
overall_ci_upper = prev_results['overall']['ci_upper']

print(f"\nOverall diabetes prevalence: {overall_prev:.2%}")
print(f"95% CI: [{overall_ci_lower:.2%}, {overall_ci_upper:.2%}]")

# Prepare age data for visualization
age_data = pd.DataFrame(prev_results['by_age'])

print("\nAge group data prepared:")
print(age_data[['age_group', 'prevalence', 'n']])

# Prepare sex data
sex_data = pd.DataFrame(prev_results['by_sex'])

print("\nSex data prepared:")
print(sex_data[['sex', 'prevalence', 'n']])

# Prepare race data
race_data = pd.DataFrame(prev_results['by_race'])
race_data = race_data.sort_values('prevalence', ascending=True)

print("\nRace/ethnicity data prepared:")
print(race_data[['race', 'prevalence', 'n']])

# Figure 1: Overall Prevalence with Confidence Interval
fig, ax = plt.subplots(figsize=(8, 6))

ax.barh([0], [overall_prev], color='steelblue', alpha=0.7, height=0.4)
ax.errorbar(overall_prev, 0, 
            xerr=[[overall_prev - overall_ci_lower], [overall_ci_upper - overall_prev]], 
            fmt='none', color='black', capsize=10, linewidth=2)

ax.set_yticks([0])
ax.set_yticklabels(['Overall'])
ax.set_xlabel('Prevalence (%)', fontsize=12, fontweight='bold')
ax.set_title('Diabetes Prevalence with 95% Confidence Interval', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlim(0, 0.20)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x*100:.1f}%'))

ax.text(overall_prev, 0.15, f'{overall_prev*100:.2f}%', 
        ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/figures/overall_prevalence.png', dpi=300, bbox_inches='tight')
print("\nSaved: overall_prevalence.png")
plt.close()

# Figure 2: Prevalence by Age Group
fig, ax = plt.subplots(figsize=(10, 6))

y_pos = np.arange(len(age_data))
prevalences = age_data['prevalence'].values
ci_lower = age_data['ci_lower'].values
ci_upper = age_data['ci_upper'].values

ax.barh(y_pos, prevalences, color='steelblue', alpha=0.7, height=0.6)

for i in range(len(age_data)):
    ax.errorbar(prevalences[i], y_pos[i],
                xerr=[[prevalences[i] - ci_lower[i]], [ci_upper[i] - prevalences[i]]],
                fmt='none', color='black', capsize=5, linewidth=1.5)
    
    ax.text(prevalences[i] + 0.005, y_pos[i], f'{prevalences[i]*100:.1f}%',
            va='center', fontsize=10, fontweight='bold')

ax.set_yticks(y_pos)
ax.set_yticklabels(age_data['age_group'])
ax.set_xlabel('Prevalence (%)', fontsize=12, fontweight='bold')
ax.set_ylabel('Age Group', fontsize=12, fontweight='bold')
ax.set_title('Diabetes Prevalence by Age Group', fontsize=14, fontweight='bold', pad=20)
ax.set_xlim(0, 0.20)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x*100:.0f}%'))

plt.tight_layout()
plt.savefig('../outputs/figures/prevalence_by_age.png', dpi=300, bbox_inches='tight')
print("Saved: prevalence_by_age.png")
plt.close()

# Figure 3: Prevalence by Sex
fig, ax = plt.subplots(figsize=(8, 6))

y_pos = np.arange(len(sex_data))
prevalences = sex_data['prevalence'].values
ci_lower = sex_data['ci_lower'].values
ci_upper = sex_data['ci_upper'].values

colors = ['#1f77b4', '#ff7f0e']
ax.barh(y_pos, prevalences, color=colors, alpha=0.7, height=0.5)

for i in range(len(sex_data)):
    ax.errorbar(prevalences[i], y_pos[i],
                xerr=[[prevalences[i] - ci_lower[i]], [ci_upper[i] - prevalences[i]]],
                fmt='none', color='black', capsize=5, linewidth=1.5)
    
    ax.text(prevalences[i] + 0.005, y_pos[i], f'{prevalences[i]*100:.1f}%',
            va='center', fontsize=11, fontweight='bold')

ax.set_yticks(y_pos)
ax.set_yticklabels(sex_data['sex'])
ax.set_xlabel('Prevalence (%)', fontsize=12, fontweight='bold')
ax.set_ylabel('Sex', fontsize=12, fontweight='bold')
ax.set_title('Diabetes Prevalence by Sex', fontsize=14, fontweight='bold', pad=20)
ax.set_xlim(0, 0.20)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x*100:.0f}%'))

plt.tight_layout()
plt.savefig('../outputs/figures/prevalence_by_sex.png', dpi=300, bbox_inches='tight')
print("Saved: prevalence_by_sex.png")
plt.close()

# Figure 4: Prevalence by Race/Ethnicity
fig, ax = plt.subplots(figsize=(10, 8))

y_pos = np.arange(len(race_data))
prevalences = race_data['prevalence'].values
ci_lower = race_data['ci_lower'].values
ci_upper = race_data['ci_upper'].values

ax.barh(y_pos, prevalences, color='steelblue', alpha=0.7, height=0.6)

for i in range(len(race_data)):
    ax.errorbar(prevalences[i], y_pos[i],
                xerr=[[prevalences[i] - ci_lower[i]], [ci_upper[i] - prevalences[i]]],
                fmt='none', color='black', capsize=5, linewidth=1.5)
    
    ax.text(prevalences[i] + 0.005, y_pos[i], f'{prevalences[i]*100:.1f}%',
            va='center', fontsize=10, fontweight='bold')

ax.set_yticks(y_pos)
ax.set_yticklabels(race_data['race'])
ax.set_xlabel('Prevalence (%)', fontsize=12, fontweight='bold')
ax.set_ylabel('Race/Ethnicity', fontsize=12, fontweight='bold')
ax.set_title('Diabetes Prevalence by Race/Ethnicity', fontsize=14, fontweight='bold', pad=20)
ax.set_xlim(0, 0.20)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x*100:.0f}%'))

plt.tight_layout()
plt.savefig('../outputs/figures/prevalence_by_race.png', dpi=300, bbox_inches='tight')
print("Saved: prevalence_by_race.png")
plt.close()

# Figure 5: Combined demographic comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Age subplot
ax = axes[0]
y_pos = np.arange(len(age_data))
ax.barh(y_pos, age_data['prevalence'].values, color='steelblue', alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(age_data['age_group'], fontsize=9)
ax.set_xlabel('Prevalence', fontsize=10)
ax.set_title('By Age Group', fontsize=12, fontweight='bold')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x*100:.0f}%'))

# Sex subplot
ax = axes[1]
y_pos = np.arange(len(sex_data))
ax.barh(y_pos, sex_data['prevalence'].values, color=['#1f77b4', '#ff7f0e'], alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(sex_data['sex'], fontsize=10)
ax.set_xlabel('Prevalence', fontsize=10)
ax.set_title('By Sex', fontsize=12, fontweight='bold')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x*100:.0f}%'))

# Race subplot
ax = axes[2]
y_pos = np.arange(len(race_data))
ax.barh(y_pos, race_data['prevalence'].values, color='steelblue', alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(race_data['race'], fontsize=9)
ax.set_xlabel('Prevalence', fontsize=10)
ax.set_title('By Race/Ethnicity', fontsize=12, fontweight='bold')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x*100:.0f}%'))

plt.suptitle('Diabetes Prevalence: Demographic Comparison', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/demographic_comparison.png', dpi=300, bbox_inches='tight')
print("Saved: demographic_comparison.png")
plt.close()

# Create publication-quality table
table_data = []

table_data.append(['Overall', 
                   f"{overall_prev*100:.2f}%",
                   f"[{overall_ci_lower*100:.2f}%, {overall_ci_upper*100:.2f}%]",
                   f"{prev_results['overall']['n']:,}"])

table_data.append(['', '', '', ''])
table_data.append(['AGE GROUP', '', '', ''])

for _, row in age_data.iterrows():
    table_data.append([row['age_group'],
                       f"{row['prevalence']*100:.2f}%",
                       f"[{row['ci_lower']*100:.2f}%, {row['ci_upper']*100:.2f}%]",
                       f"{row['n']:,}"])

table_data.append(['', '', '', ''])
table_data.append(['SEX', '', '', ''])

for _, row in sex_data.iterrows():
    table_data.append([row['sex'],
                       f"{row['prevalence']*100:.2f}%",
                       f"[{row['ci_lower']*100:.2f}%, {row['ci_upper']*100:.2f}%]",
                       f"{row['n']:,}"])

table_data.append(['', '', '', ''])
table_data.append(['RACE/ETHNICITY', '', '', ''])

for _, row in race_data.sort_values('prevalence', ascending=False).iterrows():
    table_data.append([row['race'],
                       f"{row['prevalence']*100:.2f}%",
                       f"[{row['ci_lower']*100:.2f}%, {row['ci_upper']*100:.2f}%]",
                       f"{row['n']:,}"])

table_df = pd.DataFrame(table_data, columns=['Category', 'Prevalence', '95% CI', 'Sample Size'])

table_output = '../outputs/tables/prevalence_summary_table.csv'
table_df.to_csv(table_output, index=False)
print(f"\nSaved: prevalence_summary_table.csv")

print("\n" + "="*70)
print(table_df.to_string(index=False))
print("="*70)

# Calculate key statistics for summary
age_range = age_data['prevalence'].max() - age_data['prevalence'].min()
race_range = race_data['prevalence'].max() - race_data['prevalence'].min()
sex_diff = abs(sex_data.iloc[0]['prevalence'] - sex_data.iloc[1]['prevalence'])

print("\n" + "="*70)
print("NOTEBOOK 3 COMPLETE: PREVALENCE ANALYSIS AND VISUALIZATION")
print("="*70)
print(f"Figures created: 5")
print(f"  - overall_prevalence.png")
print(f"  - prevalence_by_age.png")
print(f"  - prevalence_by_sex.png")
print(f"  - prevalence_by_race.png")
print(f"  - demographic_comparison.png")
print(f"\nTables created: 1")
print(f"  - prevalence_summary_table.csv")
print(f"\nKey Disparities:")
print(f"  Age range: {age_range*100:.1f} percentage points")
print(f"  Racial disparity: {race_range*100:.1f} percentage points")
print(f"  Sex difference: {sex_diff*100:.1f} percentage points")
print("="*70)

/var/folders/5l/3j7qr5tj6yl6_b8d9rn0grzr0000gn/T/ipykernel_49991/1026779189.py:5: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Environment loaded successfully
Loaded: 10,000 respondents
Weighted prevalence results loaded
Output directories ready

Overall diabetes prevalence: 12.55%
95% CI: [12.06%, 13.04%]

Age group data prepared:
  age_group  prevalence     n
0     18-24    0.122335  1663
1     25-34    0.125738  1753
2     35-44    0.128895  1664
3     45-54    0.129529  1623
4     55-64    0.126482  1662
5       65+    0.120093  1635

Sex data prepared:
      sex  prevalence     n
0    Male    0.130440  4903
1  Female    0.120784  5097

Race/ethnicity data prepared:
              race  prevalence     n
6  Native American    0.113385  1247
5            White    0.114230  1253
4         Hispanic    0.119220  1279
3            Other    0.123232  1252
2            Black    0.125680  1234
1            Other    0.127688  1221
0            Asian    0.135590  1296

Saved: overall_prevalence.png
Saved: prevalence_by_age.png
Saved: prevalence_by_sex.png
Saved: prevalence_by_race.png
Saved: demographic_comparison.png